# 📗 부록: 제너레이터(`yield`) 기초

> **수업 시간에 다루지 않는 참고 자료입니다.** 완주 기준에 들어가지 않습니다. `yield` 가 눈에 걸릴 때 열어 보세요.

23일차 챗봇 코드(`core/chatbot_core.py`)에 이런 함수가 있습니다.

```python
def stream_reply(message, history):
    for token in chain.stream(...):
        yield token
```

`return` 이 아니라 **`yield`** 입니다. 이 부록은 그 차이만 짚습니다.


## 0. 제너레이터란 무엇이고, 언제 쓰나

**제너레이터**는 값을 한꺼번에 만들어 돌려주는 대신, **필요할 때 하나씩 만들어 내보내는 함수**입니다.
`return` 대신 `yield` 를 쓰면 그 함수가 제너레이터가 됩니다.

일반 함수는 "다 만들어서 한 번에 건네주는" 방식이고, 제너레이터는 "만드는 족족 건네주는" 방식입니다.

![리스트 방식과 제너레이터 비교](images/generator_concept.jpg)

### 이럴 때 씁니다

| 실무 상황 | 리스트로 하면 | 제너레이터로 하면 |
| --- | --- | --- |
| **LLM 답변을 화면에 보여 준다** | 답이 다 나올 때까지 빈 화면 | 나오는 대로 타이핑되듯 표시 |
| **수 GB 로그 파일에서 에러만 찾는다** | 파일 전체를 메모리에 올리다 죽는다 | 한 줄씩 읽어 흘려보낸다 |
| **API 를 페이지 단위로 받아 온다** | 100페이지를 다 받고 시작 | 필요한 만큼만 요청하고 멈춘다 |
| **DB 수백만 행을 배치로 처리한다** | 전부 fetch 해서 메모리 초과 | 덩어리로 꺼내 처리하고 버린다 |
| **읽기 → 정제 → 필터를 이어 붙인다** | 단계마다 중간 리스트가 쌓인다 | 중간 리스트 없이 한 줄씩 통과 |
| **센서·실시간 스트림을 받는다** | 끝이 없어 리스트로 담을 수 없다 | 도착하는 대로 처리 |

공통점은 둘입니다. **다 끝나기를 기다리지 않아도 되고, 전부 메모리에 올리지 않아도 됩니다.**

### 이럴 때는 그냥 리스트를 씁니다

- 결과가 작고, **여러 번 다시 봐야** 할 때 (제너레이터는 한 번 꺼내면 소진됩니다)
- 개수를 세거나 정렬하는 등 **전체가 한꺼번에** 필요할 때

한 줄로 줄이면 이렇습니다. **기다림을 줄이고 메모리를 아끼는 대신, 한 번만 읽을 수 있다.**


## 1. `return` 은 한 번에, `yield` 는 하나씩

`return` 은 **다 만든 뒤 한 덩어리로** 돌려줍니다. `yield` 는 **하나 만들 때마다 즉시** 넘겨줍니다.

챗봇이 답을 한 글자씩 흘려보낼 수 있는 이유가 이것입니다. 문장이 다 완성될 때까지 기다리지 않습니다.


In [ ]:
import time


def make_list(n):
    """다 만든 뒤 한꺼번에 돌려준다."""
    result = []
    for i in range(n):
        time.sleep(0.2)          # 한 조각 만드는 데 걸리는 시간을 흉내
        result.append(f"조각{i}")
    return result


def make_stream(n):
    """하나 만들 때마다 바로 넘긴다."""
    for i in range(n):
        time.sleep(0.2)
        yield f"조각{i}"          # 여기서 값을 넘기고 함수는 '멈춰서 기다린다'


start = time.time()
for x in make_list(3):
    # 첫 조각을 보기까지 0.6초를 다 기다린다
    print(f"{time.time() - start:.1f}초  {x}")

print("---")

start = time.time()
for x in make_stream(3):
    # 0.2초마다 하나씩 도착한다
    print(f"{time.time() - start:.1f}초  {x}")


위 출력에서 보이듯, 리스트 방식은 **0.6초를 기다린 뒤 세 개가 한꺼번에** 나오고,
제너레이터 방식은 **0.2초마다 하나씩** 나옵니다. 총 시간은 같지만 **첫 조각이 도착하는 시점**이 다릅니다.


## 2. 호출해도 실행되지 않는다

함수 안에 `yield` 가 하나라도 있으면, 그 함수는 호출해도 **몸통이 실행되지 않습니다**.
대신 **제너레이터 객체**를 돌려줍니다. 값을 꺼낼 때 비로소 조금씩 실행됩니다.


In [ ]:
def counter():
    print("  (몸통 시작)")
    yield 1
    print("  (1 다음으로 진행)")
    yield 2
    print("  (몸통 끝)")


gen = counter()            # 아직 아무것도 출력되지 않는다
print("호출 직후:", type(gen).__name__)

print("첫 next:", next(gen))    # 여기서 처음 '(몸통 시작)' 이 찍힌다
print("둘째 next:", next(gen))


## 3. 꺼내 쓰는 방법

보통은 `next()` 를 직접 부르지 않고 **`for` 문**에 넘깁니다. `list()` 로 한 번에 모을 수도 있습니다.


In [ ]:
def squares(n):
    for i in range(n):
        yield i * i


for v in squares(5):
    print(v, end=" ")
print()

print(list(squares(5)))          # 전부 모아 리스트로
print(sum(squares(5)))           # 합계처럼 하나씩 소비하는 함수에 바로 넘길 수도 있다


## 4. 한 번 쓰면 끝난다

제너레이터는 **소진**됩니다. 끝까지 꺼낸 뒤 다시 돌리면 아무것도 나오지 않습니다.
다시 쓰려면 함수를 **다시 호출**해 새 제너레이터를 만들어야 합니다.


In [ ]:
gen = squares(3)
print("처음:", list(gen))
print("두 번째:", list(gen))      # 이미 다 꺼내서 비어 있다
print("다시 만들면:", list(squares(3)))


이것이 챗봇 코드에서 중요한 이유가 있습니다. `st.write_stream()` 은 제너레이터를 받아 화면에 흘려보내면서
**완성된 전체 문자열을 반환**합니다. 그 반환값을 대화 이력에 저장하면 됩니다.
같은 제너레이터를 두 번 읽으려 하면 두 번째는 비어 있습니다.


## 5. 리스트 컴프리헨션과 제너레이터 표현식

`for` 문으로 함수를 만들지 않아도 제너레이터를 만들 수 있습니다.
**대괄호 `[]` 를 소괄호 `()` 로 바꾸기만** 하면 됩니다. 이것을 **제너레이터 표현식**이라고 합니다.


In [ ]:
squares_list = [i * i for i in range(5)]     # 리스트 컴프리헨션: 값 5개를 지금 다 만든다
squares_gen = (i * i for i in range(5))      # 제너레이터 표현식: 아직 아무것도 안 만들었다

print("리스트    :", squares_list)
print("제너레이터 :", squares_gen)            # 값이 아니라 제너레이터 객체가 찍힌다
print("꺼내 보면  :", list(squares_gen))      # 이때 비로소 계산된다


### `sum` · `max` · `any` · `all` 에는 제너레이터를 넘긴다

이 네 함수는 값을 **하나씩 받아 처리**합니다. 리스트를 만들어 넘기면, 쓰지도 않을 리스트가
통째로 메모리에 올라갑니다. 결과는 똑같습니다.

| 함수 | 하는 일 | 끝까지 보나 |
| --- | --- | --- |
| `sum(...)` | 전부 더한다 | 끝까지 본다 |
| `max(...)` / `min(...)` | 가장 큰(작은) 값 | 끝까지 본다 |
| `any(...)` | 하나라도 참인가 | **참을 만나면 멈춘다** |
| `all(...)` | 전부 참인가 | **거짓을 만나면 멈춘다** |


In [ ]:
import tracemalloc

# 1) 리스트를 만들어 넘기는 경우
tracemalloc.start()
started = time.perf_counter()
total_list = sum([i * i for i in range(1_000_000)])   # 대괄호: 100만 개 리스트를 먼저 만든다
list_sec = time.perf_counter() - started
list_peak = tracemalloc.get_traced_memory()[1]        # 그동안 쓴 최대 메모리
tracemalloc.stop()

# 2) 제너레이터를 넘기는 경우
tracemalloc.start()
started = time.perf_counter()
total_gen = sum(i * i for i in range(1_000_000))      # 소괄호: 리스트 없이 하나씩 더한다
gen_sec = time.perf_counter() - started
gen_peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"리스트     합계 {total_list:,}  {list_sec:.2f}초  최대 메모리 {list_peak / 1024 / 1024:6.1f} MB")
print(f"제너레이터  합계 {total_gen:,}  {gen_sec:.2f}초  최대 메모리 {gen_peak / 1024:6.1f} KB")
print("두 합계가 같은가:", total_list == total_gen)


합계는 완전히 같고 **메모리만 크게 다릅니다**. 쓰고 바로 버릴 리스트를 만들지 않았기 때문입니다.

`sum(i * i for i in range(1_000_000))` 처럼 함수의 인자가 제너레이터 표현식 **하나뿐이면
괄호를 겹쳐 쓰지 않아도** 됩니다. `sum((i * i for i in ...))` 로 써도 같습니다.


#### `any` 와 `all` 은 답이 정해지면 거기서 멈춘다

`any` 는 참을 하나 만나면, `all` 은 거짓을 하나 만나면 **나머지를 보지 않고 끝냅니다**.
그런데 대괄호로 리스트를 먼저 만들면 **이미 전부 계산한 뒤**라 이 이점이 사라집니다.

아래는 값을 검사할 때마다 그 값을 찍어 무엇을 실제로 봤는지 드러냅니다.


In [ ]:
def checked(numbers):
    """검사한 값을 찍으며 하나씩 흘려보낸다. 어디서 멈추는지 보려는 용도."""
    for n in numbers:
        print(f"    검사: {n}")
        yield n


nums = [1, 3, 5, 8, 9, 11]      # 짝수는 8 하나뿐이다

print("[제너레이터] any(...)  소괄호")
print("  결과:", any(n % 2 == 0 for n in checked(nums)))

print("[리스트]     any([...])  대괄호")
print("  결과:", any([n % 2 == 0 for n in checked(nums)]))


제너레이터 쪽은 **8에서 멈춰 4개만** 검사했고, 리스트 쪽은 **6개를 전부** 검사했습니다.
답은 같지만 한 일의 양이 다릅니다. 검사 하나가 무거울수록(파일 읽기·API 호출) 차이가 커집니다.

`all` 도 같습니다. 거짓을 하나 만나면 그 자리에서 끝냅니다.


In [ ]:
print("[제너레이터] all(...)  세 번째 값이 음수")
print("  결과:", all(n > 0 for n in checked([5, 7, -1, 9, 11])))


### 둘의 차이

| | 리스트 컴프리헨션 `[...]` | 제너레이터 표현식 `(...)` |
| --- | --- | --- |
| 값을 만드는 시점 | 그 줄에서 **전부** | 꺼낼 때 **하나씩** |
| 메모리 | 전체 크기만큼 | 한 개 분량 |
| 다시 쓰기 | 몇 번이든 | **한 번**(소진) |
| `len()` · 인덱싱 `[0]` | 된다 | 안 된다 |
| 어울리는 곳 | 결과를 두고두고 쓸 때 | `sum`·`for` 처럼 한 번만 훑을 때 |

고르는 기준은 하나입니다. **그 값을 다시 볼 일이 있으면 리스트, 한 번 훑고 버릴 것이면 제너레이터.**


## 6. 메모리: 다 만들지 않는다

리스트는 모든 값을 메모리에 올립니다. 제너레이터는 **지금 꺼내는 값 하나만** 들고 있습니다.


In [ ]:
import sys

nums_list = [i for i in range(100_000)]          # 리스트
nums_gen = (i for i in range(100_000))           # 제너레이터 표현식(대괄호 대신 소괄호)

print("리스트   :", f"{sys.getsizeof(nums_list):,} 바이트")
print("제너레이터:", f"{sys.getsizeof(nums_gen):,} 바이트")


## 7. `yield from` : 다른 제너레이터를 그대로 흘려보내기

제너레이터가 만든 값을 그대로 넘길 때는 `for` + `yield` 대신 `yield from` 한 줄로 씁니다.


In [ ]:
def inner():
    yield "가"
    yield "나"


def outer_long():
    for x in inner():        # 하나씩 받아서 다시 넘긴다
        yield x


def outer_short():
    yield from inner()       # 위와 같은 뜻


print(list(outer_long()), list(outer_short()))


## 8. 실무에서 쓰는 모양

여기까지가 문법입니다. 이제 앞에서 본 것들(`yield` · `islice` 로 필요한 만큼만 꺼내기)을
합쳐 실제로 쓰는 모양을 봅니다.


### 큰 로그 파일에서 에러 몇 건만 꺼내기

파일을 한 줄씩 읽는 제너레이터와, 에러만 거르는 제너레이터를 **이어 붙입니다**.
중간 리스트가 하나도 생기지 않고, 필요한 만큼 읽으면 거기서 멈춥니다.


In [ ]:
import itertools
import tempfile
from pathlib import Path

# 실습용 로그 파일을 임시로 만든다. 실무에서는 이미 수 GB 짜리가 있다고 생각하세요.
log_path = Path(tempfile.gettempdir()) / "access.log"
log_path.write_text(
    "\n".join(
        f"2026-08-19 10:{i // 60:02d}:{i % 60:02d} {'ERROR' if i % 137 == 0 else 'INFO'} 요청 {i}"
        for i in range(100_000)
    ),
    encoding="utf-8",
)


def read_lines(path):
    """파일을 한 줄씩 흘려보낸다. 파일 전체를 메모리에 올리지 않는다."""
    with path.open(encoding="utf-8") as f:
        for line in f:
            yield line.rstrip()


def only_errors(lines):
    """ERROR 가 든 줄만 통과시킨다."""
    for line in lines:
        if "ERROR" in line:
            yield line


# 읽기 → 거르기 를 이어 붙인다. 아직 파일을 열지도 않은 상태다.
errors = only_errors(read_lines(log_path))

# islice 로 앞의 3건만 꺼낸다. 나머지 줄은 아예 읽지 않는다.
for line in itertools.islice(errors, 3):
    print(line)

print(f"\n파일 크기: {log_path.stat().st_size // 1024:,} KB (전부 읽지 않았다)")


## 9. 23일차 코드와 이어 보기

`core/chatbot_core.py` 의 구조는 결국 이렇습니다.

```python
def stream_reply(message, history):
    ...
    for token in chain.stream({"history": past, "input": message}):
        yield token          # 모델이 뱉는 조각을 그대로 흘려보낸다
```

- 모델이 답을 만드는 대로 조각(token)이 `yield` 로 나갑니다.
- Streamlit 앱은 그것을 `st.write_stream()` 에 그대로 넘깁니다.
- 그래서 답변이 **타이핑되듯** 나타납니다.

`reply()` 함수는 그 조각을 전부 이어 붙인 버전입니다.

```python
def reply(message, history):
    return "".join(stream_reply(message, history))
```


## 정리

- `yield` 가 있는 함수는 호출해도 실행되지 않고 **제너레이터**를 돌려준다
- 값을 꺼낼 때마다 **거기까지만** 실행되고 멈춘다
- **한 번 소진하면** 다시 쓸 수 없다. 다시 호출해 새로 만든다
- 전부 만들지 않으므로 **메모리를 적게** 쓰고, **첫 값이 빨리** 나온다
- 대괄호 `[...]` 를 소괄호 `(...)` 로 바꾸면 **제너레이터 표현식**이다. `sum`·`max`·`any`·`all`·`for` 처럼 한 번만 훑는 자리에는 리스트를 만들지 않는다. 특히 `any`·`all` 은 답이 정해지면 **거기서 멈춘다**
- 스트리밍 챗봇이 `yield` 를 쓰는 이유가 바로 이 두 가지다
